# Assignment 06 - Project: Deep Research with LangGraph - Multi-Agent Research System
This notebook implements **Module 6: Multi-Agent Research System - Full Multi-Agent Research System**. We combine user scoping (with interrupts), supervisor-directed research, mock MCP resources, compilation, and a final review loop into a single master system.

In [1]:
import sys
import os
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))
from mock_llm import get_llm

load_dotenv()
print("Environment loaded successfully.")

Environment loaded successfully.


In [2]:
from typing import List, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class FullSystemState(TypedDict):
    # Scoping
    topic: str
    questions: List[str]
    answers: List[str]
    brief: str
    # Supervisor & Research
    notes: str
    next_agent: str
    report: str

In [3]:
# Node 1: Scoping questions
def scoping_ask(state: FullSystemState):
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = f"Draft 3 clarifying questions for research topic: {state['topic']}"
    # Simulating simple output extraction
    return {"questions": [
        "Should we focus on asymmetric algorithms?",
        "What is the timeline scope?",
        "Who is the target audience?"
    ]}

In [4]:
# Node 2: Scoping Brief
def scoping_brief(state: FullSystemState):
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = f"Write a brief for {state['topic']} based on answers: {state['answers']}"
    res = llm.invoke(prompt)
    return {"brief": res.content, "notes": "Initial briefing details set."}

In [5]:
# Node 3: Supervisor
def supervisor(state: FullSystemState):
    # Simple supervisor selection logic to simulate multi-agent supervisor
    if "market" not in state["notes"].lower():
        next_agent = "market_researcher"
    elif "technical" not in state["notes"].lower():
        next_agent = "technical_researcher"
    else:
        next_agent = "compile_report"
    return {"next_agent": next_agent}

In [6]:
# Node 4: Market Researcher
def market_researcher(state: FullSystemState):
    new_notes = state["notes"] + "\n[Market Findings] Business migrations to post-quantum standards will double by 2026."
    return {"notes": new_notes}

In [7]:
# Node 5: Tech Researcher
def technical_researcher(state: FullSystemState):
    new_notes = state["notes"] + "\n[Technical Findings] NIST has finalized standard specifications for ML-KEM post-quantum keys."
    return {"notes": new_notes}

In [8]:
# Node 6: Report Compiler
def report_compiler(state: FullSystemState):
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = f"Compile final report from notes: {state['notes']}"
    res = llm.invoke(prompt)
    return {"report": res.content}

In [9]:
def route_next(state: FullSystemState):
    return state["next_agent"]

In [10]:
# Build master graph
builder = StateGraph(FullSystemState)
builder.add_node("scoping_ask", scoping_ask)
builder.add_node("scoping_brief", scoping_brief)
builder.add_node("supervisor", supervisor)
builder.add_node("market_researcher", market_researcher)
builder.add_node("technical_researcher", technical_researcher)
builder.add_node("compile_report", report_compiler)

builder.add_edge(START, "scoping_ask")
builder.add_edge("scoping_ask", "scoping_brief")
builder.add_edge("scoping_brief", "supervisor")

builder.add_conditional_edges(
    "supervisor",
    route_next,
    {
        "market_researcher": "market_researcher",
        "technical_researcher": "technical_researcher",
        "compile_report": "compile_report"
    }
)
builder.add_edge("market_researcher", "supervisor")
builder.add_edge("technical_researcher", "supervisor")
builder.add_edge("compile_report", END)

# Scoping pauses before compiling brief to verify user scoping answers
memory = MemorySaver()
graph = builder.compile(checkpointer=memory, interrupt_before=["scoping_brief"])

In [11]:
# Print the graph architecture
try:
    print(graph.get_graph().draw_ascii())
except Exception as e:
    print("Could not draw graph:", e)

                                     +-----------+                                     
                                     | __start__ |                                     
                                     +-----------+                                     
                                           *                                           
                                           *                                           
                                           *                                           
                                    +-------------+                                    
                                    | scoping_ask |                                    
                                    +-------------+                                    
                                           *                                           
                                           *                                           
                                

In [12]:
thread_config = {"configurable": {"thread_id": "thread-master"}}
initial_state = {
    "topic": "quantum threats to cryptography",
    "questions": [],
    "answers": [],
    "brief": "",
    "notes": "",
    "next_agent": "",
    "report": ""
}

print("--- Step 1: Initiating Full System Scoping Questions ---")
for event in graph.stream(initial_state, thread_config, stream_mode="values"):
    if "questions" in event and event["questions"]:
        print("\nQuestions generated:")
        for idx, q in enumerate(event["questions"], 1):
            print(f"{idx}. {q}")

--- Step 1: Initiating Full System Scoping Questions ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---

Questions generated:
1. Should we focus on asymmetric algorithms?
2. What is the timeline scope?
3. Who is the target audience?


In [13]:
print("--- Step 2: Answering and Running Supervisor Researchers ---")
graph.update_state(thread_config, {"answers": [
    "Yes, asymmetric algorithms are primary threat.",
    "Timeline scope is 5 years.",
    "Targeting IT security executives."
]})

res = graph.invoke(None, thread_config)
print("\n--- Final Multi-Agent Compiled Report ---")
print(res["report"])

--- Step 2: Answering and Running Supervisor Researchers ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---



--- Final Multi-Agent Compiled Report ---
# Final Report – Post‑Quantum Cryptography Landscape  
**Prepared for:** [Client / Internal Stakeholders]  
**Date:** 12 July 2026  

---

## 1. Executive Summary  

The post‑quantum (PQ) transition is accelerating. Market analysis indicates that **business migrations to PQ standards will double by the end of 2026**, driven by regulatory pressure, emerging quantum‑capable threats, and growing vendor readiness. On the technical front, **NIST has officially finalized the specification for ML‑KEM (Module‑Lattice‑based Key Encapsulation Mechanism)**, cementing the first standardised PQ key‑exchange primitive for wide‑scale deployment.  

Together, these developments create a narrow window for organizations to mature their PQ strategy, secure supply‑chain readiness, and capture competitive advantage. This report consolidates the latest market and technical intelligence and outlines actionable recommendations.

---

## 2. Market Findings  

| Metric